In [ ]:
!pip install opencv-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Konfiguration Interessensgebiet (ROI) und BEV
DATA_DIR      = "data/"
ROI_TL        = (0.38, 0.54)
ROI_TR        = (0.62, 0.54)
ROI_BR        = (1.00, 1.00)
ROI_BL        = (0.0001, 1.00)
BEV_W, BEV_H  = 400, 520

# Filterparameter
BLUR_KSIZE    = (5, 5)     
TOPHAT_WIDTH  = 41         
LANE_THRESH   = 20         

# Parameter des Gleitfensters
N_WINDOWS     = 9
MARGIN        = 40
MINPIX        = 30

def process_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Error loading {img_path}")
        return
    
    h, w = img.shape

    # 1. Perspektivverzerrung
    src_pts = np.float32([
        [ROI_TL[0]*w, ROI_TL[1]*h],
        [ROI_TR[0]*w, ROI_TR[1]*h],
        [ROI_BR[0]*w, ROI_BR[1]*h],
        [ROI_BL[0]*w, ROI_BL[1]*h],
    ])
    dst_pts = np.float32([
        [0,      0     ],
        [BEV_W,  0     ],
        [BEV_W,  BEV_H ],
        [0,      BEV_H ],
    ])

    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    Minv = cv2.getPerspectiveTransform(dst_pts, src_pts)

    bev_raw = cv2.warpPerspective(img, M, (BEV_W, BEV_H))

    roi_vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    pts_i = src_pts.astype(np.int32).reshape((-1,1,2))
    cv2.polylines(roi_vis, [pts_i], True, (0, 210, 255), 2)

    # 2. Asymmetrischer Zylinder
    bev_blur = cv2.GaussianBlur(bev_raw, BLUR_KSIZE, 0)
    kernel_1d = cv2.getStructuringElement(cv2.MORPH_RECT, (TOPHAT_WIDTH, 1))
    bev_tophat = cv2.morphologyEx(bev_blur, cv2.MORPH_TOPHAT, kernel_1d)
    _, bev_binary = cv2.threshold(bev_tophat, LANE_THRESH, 255, cv2.THRESH_BINARY)

    # 3. Schiebefenster
    histogram = np.sum(bev_binary[bev_binary.shape[0]//2:, :], axis=0)
    midpoint = int(histogram.shape[0]//2)
    leftx_base = np.argmax(histogram[:midpoint])
    rightx_base = np.argmax(histogram[midpoint:]) + midpoint

    window_height = int(bev_binary.shape[0]//N_WINDOWS)
    nonzero = bev_binary.nonzero()
    nonzeroy = np.array(nonzero[0])
    nonzerox = np.array(nonzero[1])

    leftx_current = leftx_base
    rightx_current = rightx_base

    left_lane_inds = []
    right_lane_inds = []
    
    # Erstelle ein RGB-Bild zur Visualisierung
    out_img = np.dstack((bev_binary, bev_binary, bev_binary))

    for window in range(N_WINDOWS):
        win_y_low = bev_binary.shape[0] - (window+1)*window_height
        win_y_high = bev_binary.shape[0] - window*window_height
        win_xleft_low = leftx_current - MARGIN
        win_xleft_high = leftx_current + MARGIN
        win_xright_low = rightx_current - MARGIN
        win_xright_high = rightx_current + MARGIN
        
        cv2.rectangle(out_img,(win_xleft_low,win_y_low),(win_xleft_high,win_y_high),(0,255,0), 2) 
        cv2.rectangle(out_img,(win_xright_low,win_y_low),(win_xright_high,win_y_high),(0,255,0), 2) 
        
        good_left_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) & 
        (nonzerox >= win_xleft_low) &  (nonzerox < win_xleft_high)).nonzero()[0]
        good_right_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) & 
        (nonzerox >= win_xright_low) &  (nonzerox < win_xright_high)).nonzero()[0]
        
        left_lane_inds.append(good_left_inds)
        right_lane_inds.append(good_right_inds)
        
        if len(good_left_inds) > MINPIX:
            leftx_current = int(np.mean(nonzerox[good_left_inds]))
        if len(good_right_inds) > MINPIX:        
            rightx_current = int(np.mean(nonzerox[good_right_inds]))

    left_lane_inds = np.concatenate(left_lane_inds)
    right_lane_inds = np.concatenate(right_lane_inds)

    leftx = nonzerox[left_lane_inds]
    lefty = nonzeroy[left_lane_inds] 
    rightx = nonzerox[right_lane_inds]
    righty = nonzeroy[right_lane_inds] 

    out_img[lefty, leftx] = [255, 0, 0]
    out_img[righty, rightx] = [0, 0, 255]

    try:
        left_fit = np.polyfit(lefty, leftx, 2)
        right_fit = np.polyfit(righty, rightx, 2)
        polyfit_success = True
    except TypeError:
        polyfit_success = False

    # 4. Zurückziehen und zurückspulen
    final = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    if polyfit_success:
        ploty = np.linspace(0, bev_binary.shape[0]-1, bev_binary.shape[0])
        left_fitx = left_fit[0]*ploty**2 + left_fit[1]*ploty + left_fit[2]
        right_fitx = right_fit[0]*ploty**2 + right_fit[1]*ploty + right_fit[2]

        color_warp = np.zeros_like(out_img).astype(np.uint8)
        pts_left = np.array([np.transpose(np.vstack([left_fitx, ploty]))])
        pts_right = np.array([np.flipud(np.transpose(np.vstack([right_fitx, ploty])))])
        pts = np.hstack((pts_left, pts_right))

        cv2.fillPoly(color_warp, np.int_([pts]), (0, 255, 0))
        cv2.polylines(color_warp, np.int_([pts_left]), False, (255, 0, 0), 10)
        cv2.polylines(color_warp, np.int_([pts_right]), False, (0, 0, 255), 10)

        newwarp = cv2.warpPerspective(color_warp, Minv, (w, h)) 
        final = cv2.addWeighted(final, 1, newwarp, 0.3, 0)

    # 5. Visualisierungseinrichtung
    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Processing: {os.path.basename(img_path)}", fontsize=16, fontweight='bold')
    
    axs[0, 0].imshow(roi_vis[..., ::-1])
    axs[0, 0].set_title("1. Original Image + ROI")
    
    axs[0, 1].imshow(bev_raw, cmap='gray')
    axs[0, 1].set_title("2. BEV Raw View")
    
    axs[0, 2].imshow(bev_tophat, cmap='gray')
    axs[0, 2].set_title("3. 1D Top-Hat Result")
    
    axs[1, 0].imshow(bev_binary, cmap='gray')
    axs[1, 0].set_title("4. Binary Threshold")
    
    axs[1, 1].imshow(out_img[..., ::-1])
    axs[1, 1].set_title("5. Sliding Window Detection")
    
    axs[1, 2].imshow(final[..., ::-1])
    axs[1, 2].set_title("6. Final Output")
    
    for ax in axs.flatten():
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()


image_paths = glob.glob(os.path.join(DATA_DIR, "*.jpg"))

if not image_paths:
    print(f"Nein, nein, fügen Sie Bilder hinzu, wenn Sie Ergebnisse erzielen wollen.")
else:
    for path in sorted(image_paths):
        process_image(path)